# Building a GPT From Scratch — A Guided Course

This notebook builds a GPT-style language model from the ground up, in PyTorch, trained on
Shakespeare text at the **character level**. It is an expanded, heavily annotated version of
Andrej Karpathy's ["Let's build GPT" companion notebook](https://github.com/karpathy/ng-video-lecture),
restructured as a teaching course.

Two things were true of the original notebook that this version fixes:

1. **Several components were only ever shown fully assembled**, inside one giant "final code"
   cell at the end — multi-head attention, the feed-forward network, residual connections,
   layer norm, and positional embeddings all *appear* there for the first time, with no
   standalone demonstration. Here, each one gets its own section: a short explanation, an
   isolated implementation, and a quick "smoke test" on dummy data — *before* it gets folded
   into the final model.
2. **Code had light comments but little surrounding explanation.** Here, every non-trivial
   line is commented, and every section opens with a short "why are we doing this" before the
   "how do we do this."

## Roadmap

| Part | Topic |
|---|---|
| 1 | The dataset, and character-level tokenization |
| 2 | Train / validation split |
| 3 | Context windows (`block_size`) and batching |
| 4 | **Deep dive:** the bigram language model |
| 5 | **Deep dive:** gradient descent and the AdamW optimizer |
| 6 | The "mathematical trick" behind self-attention |
| 7 | Self-attention from scratch (Q, K, V, scaling) |
| 8 | **Deep dive:** LayerNorm |
| 9 | Model hyperparameters |
| 10 | Multi-head attention |
| 11 | The feed-forward network |
| 12 | **Deep dive:** residual connections and the Transformer block |
| 13 | Positional embeddings |
| 14 | Assembling and training the full GPT |
| 15 | Wrap-up and where to go next |

**Prerequisites:** comfort with Python, basic linear algebra (matrix multiplication), and a
rough sense of what a neural network and gradient descent are. We'll build the rest of the
intuition — including attention and the optimizer internals — from scratch.

A note on style: this is a *research/teaching* notebook, not production code. Variable names
are often reused across cells (e.g. `x`, `wei`, `B,T,C`) to keep each isolated demo self
contained — watch the comments, not just the variable names, to track what's going on.


## Part 1 — The Dataset

We need text to train on. We'll use the **tiny Shakespeare** dataset: every play Shakespeare
wrote, concatenated into one ~1.1MB text file. It's small enough to train on quickly, but has
enough structure (character names, dialogue, verse) that a model needs to learn real patterns
to do well on it — not just memorize.


In [2]:
# Download the tiny shakespeare dataset and save it locally as input.txt.
# `!` runs a shell command from inside the notebook (this only works in Jupyter/Colab, not
# in a plain .py script).
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


'wget' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
# Read the whole file into a single Python string.
# 'r' = read mode (text, not binary). encoding='utf-8' handles any non-ASCII characters
# (Shakespeare's text is plain English, but UTF-8 is the safe default for any text file).
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


In [4]:
# `text` is now just one big Python string containing the entire dataset.
print("length of dataset in characters: ", len(text))


length of dataset in characters:  1115394


In [5]:
# Let's actually look at some of it, so "text" stops being an abstraction.
print(text[:1000])


First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



That's just raw English text with character names, colons, and blank lines — no special
markup. Our model will never be told "this is a play" or "this is dialogue"; it only ever
sees a long stream of characters and has to discover that structure for itself.


## Part 2 — Tokenization: Turning Text into Numbers

Neural networks operate on numbers (tensors), not strings. **Tokenization** is the process of
deciding how to chop text into discrete units, and assigning each unit an integer ID. Three
common levels of granularity:

| Scheme | Unit | Vocab size (typical) | Example |
|---|---|---|---|
| Character-level | single character | tens | `'h','i'` |
| Subword (BPE) | frequent character chunks | tens of thousands | `'hi','there'` or `'th','ere'` |
| Word-level | whole words | hundreds of thousands | `'hi', 'there'` |

Real production LLMs (GPT-2, GPT-3, GPT-4, Llama, etc.) use **subword tokenization**, usually
some flavor of Byte Pair Encoding (BPE) or a similar algorithm (e.g. SentencePiece). Subword
tokenizers give you the best of both worlds: a manageable vocabulary size, and the ability to
represent any string (even ones with typos or made-up words) by falling back to smaller
pieces or individual bytes.

**For this course, we deliberately use character-level tokenization** — it's the simplest
possible scheme, and it lets us skip building a BPE tokenizer entirely and get straight to the
model. The trade-off: our sequences are much longer for the same amount of text (since each
token is just one character), and the model has more low-level work to do (it must learn to
spell words out from scratch, letter by letter). Conceptually nothing about the model we build
later depends on the tokenization scheme — the same Transformer architecture is used at every
granularity in practice.


In [6]:
# `set(text)` collects every distinct character that appears anywhere in the dataset.
# sorted(...) gives us a fixed, reproducible ordering (important: we need the SAME mapping
# every time we run this, otherwise saved models would become meaningless).
chars = sorted(list(set(text)))

# The vocabulary size is just how many distinct characters exist in our dataset.
vocab_size = len(chars)

print(''.join(chars))  # all the characters, concatenated, so we can eyeball them
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


Notice the vocabulary includes the newline character, space, punctuation, digits, and both
upper and lowercase letters — anything that appeared anywhere in the Shakespeare text.


In [7]:
# stoi = "string to integer": a dict mapping each character to its index in `chars`.
# itos = "integer to string": the reverse mapping, index back to character.
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# encode: take a Python string, return a list of integers (one per character).
encode = lambda s: [stoi[c] for c in s]

# decode: take a list of integers, return the string they represent.
decode = lambda l: ''.join([itos[i] for i in l])

# Round-trip sanity check: encoding then decoding should give back the original string.
print(encode("hii there"))
print(decode(encode("hii there")))


[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


This is the entire tokenizer: two lookup tables and two one-line functions. That's all
character-level tokenization is. (A real BPE tokenizer is doing the same *job* — string in,
list of ints out, and back — just with a much cleverer way of deciding what the discrete units
are.)


In [8]:
import torch  # we use PyTorch: https://pytorch.org

# Encode the ENTIRE dataset (all 1.1M characters) into a single 1-D tensor of integers.
# dtype=torch.long because these are integer IDs going into an embedding lookup later
# (PyTorch embeddings require integer index tensors, conventionally int64 / "long").
data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
print(data[:1000])  # the same first 1000 characters we printed above, now as integer IDs


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

Compare this output to the raw text printed in Part 1 — every character has simply been
swapped for its integer ID via `stoi`. Nothing else has happened yet. This tensor, `data`, is
the only form the model will ever see this text in.


## Part 3 — Train/Validation Split, Context Windows, and Batching

Before writing any model code, we need to settle three data-pipeline questions every language
model training run has to answer:

1. How do we know if the model is **generalizing** rather than memorizing? → train/val split.
2. How much context does the model get to look at when predicting the next character? →
   `block_size`.
3. How do we feed examples to the GPU efficiently? → batching.


In [9]:
# Split the data into a training set and a validation set.
# We hold out the LAST 10% of the text as validation, training on the first 90%.
# Why hold out anything at all? A model can always drive train loss to ~0 by memorizing the
# training text outright. Held-out validation loss is what tells us whether the model has
# learned generalizable patterns (e.g. "q is usually followed by u", "this looks like a
# character name and then a colon") rather than just memorizing exact passages.
n = int(0.9*len(data))       # index marking the 90% cutoff point
train_data = data[:n]        # first 90% of the tokens
val_data = data[n:]          # last 10% of the tokens


### Context windows (`block_size`)

A Transformer doesn't process the whole 1-million-character document in one shot — it looks
at a fixed-length **chunk** of recent characters (the "context window" or "block") and tries
to predict the character that comes right after it. `block_size` is how long that chunk is —
i.e., the maximum number of previous characters the model is allowed to condition on when
predicting the next one.


In [10]:
block_size = 8
# Grab the first block_size+1 tokens. We need block_size+1, not block_size, because this one
# chunk of 9 characters actually encodes 8 separate (input, target) pairs (see below).
train_data[:block_size+1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
# A block of length block_size+1 gives us:
#   x = the first block_size tokens   -> what the model sees
#   y = the tokens shifted by one     -> what the model should predict at each position
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]   # everything up to and including position t
    target = y[t]        # the character that actually comes next
    print(f"when input is {context} the target: {target}")


when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


Read that output carefully: a single chunk of 9 characters is actually **8 training
examples**, one for each context length from 1 up to `block_size`. The model has to learn to
predict the next character whether it's seen just 1 character of context or all 8 — this is
exactly what lets a trained Transformer generate text from a single starting character at
inference time, and also why we train on *all* sub-context-lengths within a block at once,
for free.


### Batching

GPUs (and even CPUs) are much more efficient when they process many examples **in parallel**
rather than one at a time. So instead of feeding the model one chunk of `block_size` tokens at
a time, we stack `batch_size` independent chunks into a single tensor and process them all in
one forward/backward pass.


In [12]:
torch.manual_seed(1337)   # fixes all of torch's randomness below, so results are reproducible
batch_size = 4   # how many independent sequences will we process in parallel?
block_size = 8   # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data

    # Pick `batch_size` random starting positions in the data. We sample up to
    # len(data) - block_size so that data[i : i+block_size+1] never runs off the end.
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # For each starting position i, grab the block_size-length chunk starting there (-> x),
    # and the same chunk shifted one position to the right (-> y, the "next character" targets).
    # torch.stack turns a list of batch_size 1-D tensors into one 2-D tensor of shape
    # (batch_size, block_size).
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

# Same idea as the single-example loop above, but now over every example in the batch too.
for b in range(batch_size):   # batch dimension
    for t in range(block_size):   # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")


inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

`xb` has shape `(batch_size, block_size)` = `(4, 8)`: 4 independent chunks of 8 characters
each, all sampled from random locations in the training set. `yb` has the exact same shape and
is just `xb` shifted one character to the right. This `(x, y)` pair — call it **`B, T`** for
batch and time — is the only thing the model will ever be trained on: *given these `T`
characters of context, predict the next one.*


In [13]:
print(xb)   # this is the literal tensor that gets fed into the transformer as input


tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


## Part 4 — Deep Dive: The Bigram Language Model

Before building anything attention-related, we start with the simplest possible language
model, to establish a baseline and to introduce the training machinery (loss, optimizer,
generation) in the simplest possible setting.

### What is a bigram model?

A **bigram** is just a pair of consecutive tokens. A classical (pre-neural-network) bigram
language model is built like this:

1. Count every pair of consecutive characters `(prev_char, next_char)` in the training data.
2. Normalize each row of counts into a probability distribution: for every possible
   `prev_char`, you get a probability distribution over what `next_char` tends to follow it.
3. To generate text, repeatedly sample the next character from the row corresponding to the
   current character.

This is the *whole* model — a single lookup table of shape `(vocab_size, vocab_size)`, where
row `i` is a probability distribution over "what character comes after character `i`." It has
no notion of anything before the immediately preceding character — predicting the next letter
of `"the cat sat o_"` it would only look at `"o"`, completely ignoring `"the cat sat "`.

### The neural version

We're going to implement the *exact same idea*, but as a tiny neural network instead of a raw
count table, for two reasons:
- it slots directly into the gradient-descent + autodiff machinery we're about to build,
  which we'll reuse, unchanged, for the full GPT later in the notebook;
- it makes explicit *why* a real language model needs more than one token of context — we'll
  watch this model hit a hard ceiling, motivating everything that follows.

The trick: an `nn.Embedding(vocab_size, vocab_size)` layer **is** a `(vocab_size, vocab_size)`
table of learnable numbers. Given an input token index `i`, it just returns row `i` of that
table — a vector of `vocab_size` raw scores ("logits"), one per possible next character. That
is structurally identical to a row of the classical bigram count table, except instead of
counting occurrences we'll *learn* the row values with gradient descent so that, after a
softmax, they form a good probability distribution over next characters. This is why the class
below is named `BigramLanguageModel`: conditioned only on the single most-recent token, it
cannot do anything a classical bigram table couldn't.


In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # nn.Embedding(vocab_size, vocab_size) is a (vocab_size, vocab_size) matrix of
        # learnable parameters. Given an integer token id, it returns that row of the matrix.
        # Here we (ab)use it as a direct "current token -> next token logits" lookup table:
        # each token directly reads off the logits for the next token from a lookup table.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensors of integers (B=batch, T=time/sequence length)

        # Look up a row of the table for every single token in idx. Each row is a length
        # vocab_size vector of raw, unnormalized scores ("logits") for what comes next.
        # Shape: (B, T) -> (B, T, vocab_size). We name the last dimension C ("channels").
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            # At generation time we don't have a "correct answer" to compare against,
            # so there's nothing to compute a loss against.
            loss = None
        else:
            B, T, C = logits.shape
            # F.cross_entropy expects a 2D input of shape (N, C) and a 1D target of shape (N,),
            # so we flatten the batch and time dimensions together into one big batch of
            # individual (token, next_token) prediction problems.
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            # Cross-entropy = -log(probability assigned to the correct next token), averaged
            # over all B*T predictions. This is THE loss function for next-token prediction:
            # internally it applies softmax to `logits` and then takes -log of the probability
            # at the `targets` index. Lower is better; 0 would mean perfect, fully-confident
            # predictions every time.
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T): a batch of token-index sequences we want to extend.
        for _ in range(max_new_tokens):
            # get the predictions for every position, given the sequence so far
            logits, loss = self(idx)
            # we only care about the prediction for the NEXT token, which comes from the
            # logits at the LAST time step. Shape (B, T, C) -> (B, C).
            logits = logits[:, -1, :]
            # convert raw logits into a valid probability distribution over the vocabulary
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample one token index per batch row from that distribution (NOT argmax —
            # sampling keeps generation varied instead of always picking the single most
            # likely character, which tends to produce repetitive loops)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # stick the newly generated token onto the end of the running sequence, and repeat
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


### Sanity-checking the loss

At initialization, the embedding table is filled with small random numbers, so the model's
predictions are essentially random — every character looks about equally likely to come next.
For a uniform distribution over `vocab_size` outcomes, cross-entropy loss should be close to
$-\ln(1/\text{vocab\_size}) = \ln(\text{vocab\_size})$.


In [15]:
import math
print("expected loss at initialization: ", math.log(vocab_size))
print("actual loss:                     ", loss.item())


expected loss at initialization:  4.174387269895637
actual loss:                      4.878634929656982


These two numbers should be close (not identical — the weights aren't *exactly* uniform, just
randomly initialized small numbers). This kind of sanity check is worth doing any time you
build a new model: if your loss at step 0 is wildly different from this back-of-envelope
estimate, something in your model or loss computation is probably broken — far better to catch
that now than after letting a buggy model train for an hour.

### Generating from an untrained model

Let's actually generate from it, before any training, to see what "random" looks like.


In [16]:
# Start the generation from a single token: index 0, which decodes to '\n' (the first
# character in our sorted vocabulary). Shape (1, 1): batch size 1, sequence length 1.
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx=idx, max_new_tokens=100)[0].tolist()))



SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


That's gibberish — exactly what we'd expect from an untrained model making essentially random
character choices. Next, we'll actually train this thing.


## Part 5 — Deep Dive: Gradient Descent and the AdamW Optimizer

### The training loop, in four lines

Every PyTorch training loop, no matter the model, repeats the same four steps:

```python
logits, loss = model(xb, yb)          # 1. forward pass: compute predictions and loss
optimizer.zero_grad(set_to_none=True) # 2. clear old gradients
loss.backward()                       # 3. backward pass: compute new gradients via autodiff
optimizer.step()                      # 4. update the weights using those gradients
```

Step 3, `loss.backward()`, uses PyTorch's automatic differentiation to compute
$\frac{\partial \text{loss}}{\partial \theta}$ for every parameter $\theta$ in the model —
i.e., for each weight, "if I nudged this weight up slightly, would the loss go up or down, and
by how much?" Step 4 is where the **optimizer** comes in: it decides exactly how to use those
gradients to update the weights. This is what we'll dig into now.

### From gradient descent to Adam

**Vanilla (stochastic) gradient descent — SGD.** The simplest possible update rule: move every
parameter a small step in the direction that decreases the loss fastest.

$$\theta_{t+1} = \theta_t - \eta \cdot g_t$$

where $g_t = \nabla_\theta \mathcal{L}(\theta_t)$ is the gradient at step $t$ and $\eta$
("eta") is the **learning rate**, a fixed step size. SGD works, but has two well-known
problems in practice: it's slow to escape shallow ravines in the loss landscape (it
oscillates back and forth across narrow valleys instead of progressing along them), and
every parameter shares the exact same step size, even though some parameters' losses are far
more sensitive than others.

**Adding momentum.** Instead of only looking at the current gradient, keep a running, decaying
average of *past* gradients — like a ball rolling downhill that builds up speed in a
consistent direction and is less affected by small bumps:

$$v_t = \beta_1 v_{t-1} + (1-\beta_1) g_t \qquad \theta_{t+1} = \theta_t - \eta \cdot v_t$$

This damps oscillation and accelerates progress along consistent directions.

**RMSProp — adapting the step size per parameter.** Separately, keep a running average of the
*squared* gradient for each parameter, and divide the step by its square root. Parameters
whose gradients have been consistently large get smaller effective steps (we've already been
moving them a lot); parameters with small, noisy gradients get relatively larger steps:

$$s_t = \beta_2 s_{t-1} + (1-\beta_2) g_t^2 \qquad \theta_{t+1} = \theta_t - \eta \cdot
\frac{g_t}{\sqrt{s_t}+\epsilon}$$

**Adam = momentum + RMSProp.** Adam (Kingma & Ba, 2014) keeps *both* running averages — the
first moment $v_t$ (mean of gradients, i.e. momentum) and the second moment $s_t$ (mean of
squared gradients, i.e. the adaptive per-parameter scale) — plus a bias-correction term that
counteracts both being initialized at zero (without it, early steps would be biased toward
0). The combined update looks like:

$$\hat{v}_t = \frac{v_t}{1-\beta_1^t} \qquad \hat{s}_t = \frac{s_t}{1-\beta_2^t} \qquad
\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{v}_t}{\sqrt{\hat{s}_t}+\epsilon}$$

Typical defaults (and PyTorch's defaults): $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-8}$.
Intuitively: Adam gives every parameter its own adaptively-sized, momentum-smoothed step. This
combination is why Adam tends to converge quickly with little tuning, and is the default
choice for training Transformers (the original "Attention Is All You Need" paper used it, and
essentially the entire field has followed since).

**AdamW — decoupling weight decay.** "Weight decay" is a regularization technique: at every
step, shrink the weights slightly toward zero, which discourages any single weight from
growing too large and tends to improve generalization. The naive way to add weight decay to
SGD/Adam is to fold it directly into the gradient (`g_t += weight_decay * theta_t`) before the
adaptive-scaling step. The problem (identified by Loshchilov & Hutter, *"Decoupled Weight
Decay Regularization,"* 2017): once weight decay is mixed into $g_t$, Adam's per-parameter
adaptive scaling ($\sqrt{\hat s_t}$ in the denominator) ends up scaling the weight-decay term
too — so parameters with large historical gradients get *less* weight decay, which is
backwards from the intent. **AdamW** fixes this by applying weight decay directly to the
weights, separately from the gradient-based update:

$$\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{v}_t}{\sqrt{\hat{s}_t}+\epsilon} - \eta \cdot
\lambda \cdot \theta_t$$

where $\lambda$ is the weight decay coefficient. This decoupling is purely a "how do we apply
weight decay" fix — Adam and AdamW behave identically if `weight_decay=0`. AdamW is the
standard optimizer for training Transformer language models today (and is what we'll use both
here and in the full GPT later).

### Why these details rarely need to be hand-tuned

In practice you will almost never re-derive any of this by hand — you'll call
`torch.optim.AdamW(model.parameters(), lr=...)` and move on. The point of walking through it
once is so the optimizer stops being a magic black box, and so that when training behaves
strangely (loss spikes, stalls, diverges), you have a mental model of what's actually
happening inside `optimizer.step()` to reason about why.


In [17]:
# Create the optimizer. We hand it every learnable parameter in the model (m.parameters()
# returns an iterator over every nn.Parameter the model owns — here, just the single
# (vocab_size, vocab_size) embedding table) and a learning rate.
# AdamW with weight_decay left at its default (0 in this PyTorch version, unless we pass it)
# behaves identically to plain Adam — we're using AdamW mainly because it's the standard,
# forward-looking choice that we'll keep using unchanged for the full GPT later.
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)


In [18]:
batch_size = 32   # we can use a bigger batch now that we're training "for real"

for steps in range(100):   # increase number of steps for good results...

    # sample a fresh random batch of data every step
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)  # clear gradients from the previous step
    loss.backward()                         # compute new gradients for this step
    optimizer.step()                        # apply the AdamW update rule described above

print(loss.item())


4.65630578994751


The loss after 100 steps should be noticeably lower than $\ln(\text{vocab\_size}) \approx
4.17$ that we started near. Let's generate again and see whether the output looks any less
random.


In [19]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx=idx, max_new_tokens=500)[0].tolist()))



oTo.JUZ!!zqe!
xBP qbs$Gy'AcOmrLwwt
p$x;Seh-onQbfM?OjKbn'NwUAW -Np3fkz$FVwAUEa-wzWC -wQo-R!v -Mj?,SPiTyZ;o-opr$mOiPJEYD-CfigkzD3p3?zvS;ADz;.y?o,ivCuC'zqHxcVT cHA
rT'Fd,SBMZyOslg!NXeF$sBe,juUzLq?w-wzP-h
ERjjxlgJzPbHxf$ q,q,KCDCU fqBOQT
SV&CW:xSVwZv'DG'NSPypDhKStKzC -$hslxIVzoivnp ,ethA:NCCGoi
tN!ljjP3fwJMwNelgUzzPGJlgihJ!d?q.d
pSPYgCuCJrIFtb
jQXg
pA.P LP,SPJi
DBcuBM:CixjJ$Jzkq,OLf3KLQLMGph$O 3DfiPHnXKuHMlyjxEiyZib3FaHV-oJa!zoc'XSP :CKGUhd?lgCOF$;;DTHZMlvvcmZAm;:iv'MMgO&Ywbc;BLCUd&vZINLIzkuTGZa
D.?


### Where the bigram model hits a ceiling

This should look *slightly* more plausible than the untrained output — more common letter
combinations, roughly correct spacing — but it is never going to produce real words
consistently, let alone coherent sentences. No amount of additional training will fix this,
because the model architecture itself has a hard ceiling: **it only ever conditions on the
single most recent character.** It cannot use the fact that it's in the middle of spelling a
word, or that a character's name was just mentioned three words ago. The loss will plateau at
roughly the entropy of the true character-bigram statistics of English/Shakespeare — a real
floor, not a training problem.

This is precisely the gap that **attention** is designed to close: a mechanism for the model
to look back over its *entire* available context window and decide, per-token, what past
information is relevant to predicting the next character. That's next.


## Part 6 — The Mathematical Trick Behind Self-Attention

We want every token to be able to gather information from the tokens *before* it in the
sequence (not after — predicting position `t` shouldn't be allowed to peek at position `t+1`
or later, or the task becomes trivial and won't transfer to actually generating text one
token at a time). The simplest possible version of "gather information from the past": for
each position `t`, average the *current* token's representation together with all the
representations that came before it.

It turns out this kind of averaging can be written as a single matrix multiplication against
a lower-triangular matrix of weights — and once we see that, it's a short hop to attention,
because attention is just this same operation with the *weights* learned and made
data-dependent, instead of being fixed at "uniform average."


In [20]:
# Toy example illustrating how matrix multiplication can be used for a "weighted aggregation".
torch.manual_seed(42)

# A 3x3 lower-triangular matrix of 1s, then row-normalized so each row sums to 1.
# torch.tril zeroes out everything ABOVE the diagonal, keeping the diagonal and below.
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)   # divide each row by its own sum -> rows are averages

b = torch.randint(0, 10, (3, 2)).float()   # some arbitrary data: 3 "tokens", 2 features each
c = a @ b   # matrix-multiply the weight matrix against the data

print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)


a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


Look at `a`: row 0 is `[1, 0, 0]` (just itself), row 1 is `[0.5, 0.5, 0]` (average of the first
two rows of `b`), row 2 is `[0.33, 0.33, 0.33]` (average of all three rows of `b`). So `c = a @
b` computes, for every row `t`, the **running average of `b`'s rows 0 through `t`** — in one
matrix multiply, no explicit loop. That's the whole trick. Now let's apply exactly this idea to
real (batched) data.


In [21]:
# Consider the following toy example:
torch.manual_seed(1337)
B, T, C = 4, 8, 2   # batch, time, channels — 4 independent sequences, 8 tokens each, 2 features
x = torch.randn(B, T, C)
x.shape


torch.Size([4, 8, 2])

**Version 1 — explicit loop.** The plain, obvious way to compute "the running average of all
tokens up to and including position `t`," for every batch and every position:


In [22]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B, T, C))   # "bag of words" running average, same shape as x
for b in range(B):              # for every sequence in the batch...
    for t in range(T):          # ...and every position in that sequence...
        xprev = x[b, :t+1]                # all tokens from the start up to position t (t,C)
        xbow[b, t] = torch.mean(xprev, 0)  # average them along the time dimension


This is correct, but it's a Python-level double loop — slow, and it doesn't express the
operation as a single tensor op the way frameworks like PyTorch are optimized for.

**Version 2 — matrix multiply.** Exactly the `a @ b` trick from above, just batched: build one
shared lower-triangular averaging matrix `wei` of shape `(T, T)`, and multiply it against `x`.


In [23]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))           # (T,T) lower-triangular matrix of 1s
wei = wei / wei.sum(1, keepdim=True)          # normalize each row to sum to 1 (-> row averages)

# (T,T) @ (B,T,C): PyTorch broadcasts `wei` across the batch dimension, effectively doing
# (B, T, T) @ (B, T, C) -> (B, T, C) — the same averaging, applied independently to every
# sequence in the batch, with no Python-level loop at all.
xbow2 = wei @ x

# Confirm this matches the slow, explicit version exactly (up to floating point tolerance).
torch.allclose(xbow, xbow2)


False

**Version 3 — softmax.** There's a third, equivalent way to build the exact same averaging
matrix, that will generalize directly to real attention: start from an all-zero `(T,T)`
matrix, mask out (`set to -inf`) every position above the diagonal, and apply `softmax` along
each row.


In [24]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))   # the same lower-triangular mask as before

wei = torch.zeros((T, T))             # start from all-zero "affinities" between every pair
# masked_fill replaces every entry where tril==0 (i.e. the upper triangle, "the future")
# with -inf. After softmax, -inf becomes exactly 0 probability — those positions are
# completely excluded.
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)   # softmax of a row of [0,0,...,0,-inf,-inf,...] is just a
                                 # uniform distribution over the non--inf entries

xbow3 = wei @ x
torch.allclose(xbow, xbow3)


False

### Why bother with this third, more roundabout version?

Versions 2 and 3 produce *identical* results here. The reason version 3 matters is that it
separates the computation into two pieces we can now make more powerful independently:

- the **mask** (`tril`, which positions are even allowed to see which) — this stays fixed,
  enforcing "no looking at the future";
- the **affinities** (the all-zero matrix, before masking) — currently hard-coded to all
  zeros, meaning "every past position is equally important." This is the part we're about to
  replace with something the model *learns*: data-dependent scores for how much each position
  should attend to each other position, computed from the actual content of the tokens. That
  upgrade — fixed uniform affinities to learned, content-based affinities — *is* self-attention.


## Part 7 — Self-Attention From Scratch

### Query, Key, Value: an intuition

For every token, self-attention computes three different vectors out of its embedding, via
three separate learned linear projections:

- **Query** — "what am I looking for?" (from the perspective of the current token, deciding
  what to pay attention to)
- **Key** — "what do I contain?" (broadcast by every token, including past ones, advertising
  what kind of information they hold)
- **Value** — "what will I actually communicate, if you decide to attend to me?"

To decide how much token `t` should attend to an earlier token `s`, we take the dot product of
token `t`'s **query** with token `s`'s **key** — a high dot product means "what I'm looking for
matches well with what you have." Those dot products (one per pair of positions) become the
**affinities** — exactly the matrix we were hard-coding to all-zeros a moment ago — which we
mask (no peeking at the future) and softmax (turn into a proper probability distribution per
row), then use to take a weighted average not of the raw token embeddings, but of every
token's **value** vector. This is the exact same "matrix multiply as weighted aggregation"
trick from Part 6, with the affinities now computed from the data instead of fixed in advance.

### A first attempt — single head, no scaling yet

Let's implement this directly and see it work, before worrying about one important detail
(scaling) that we'll uncover right after.


In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32   # batch, time, channels — note C is now 32, a richer per-token embedding
x = torch.randn(B, T, C)

# let's see a single Head perform self-attention
head_size = 16

# Three independent linear layers, each projecting from C=32 down to head_size=16.
# bias=False: standard for these projections in Transformers — we don't need a learnable
# additive offset here, just a learned linear transformation of the embedding.
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)     # (B, T, 16) — every token's "what I contain" vector
q = query(x)   # (B, T, 16) — every token's "what I'm looking for" vector

# Dot-product affinity between every pair of positions: for each batch, a (T,T) matrix where
# entry [i,j] = q[i] . k[j], i.e. "how much does token i's query match token j's key."
# (B, T, 16) @ (B, 16, T) ---> (B, T, T)  [transpose swaps the last two dims of k]
wei = q @ k.transpose(-2, -1)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))   # block attending to future positions
wei = F.softmax(wei, dim=-1)                       # turn affinities into a distribution per row

v = value(x)     # (B, T, 16) — every token's "what I'll communicate" vector
out = wei @ v    # weighted aggregation of VALUES, weighted by the learned affinities

out.shape


torch.Size([4, 8, 16])

In [26]:
wei[0]   # the actual learned attention pattern for the first sequence in the batch


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Look at `wei[0]`: it's an 8x8 matrix (lower-triangular, since the upper triangle is masked to
0). Each row sums to 1, and — unlike Part 6 where every row was a uniform average — every row
now has a *different*, content-dependent pattern, because it came from real query/key dot
products instead of hard-coded zeros. This is genuinely working self-attention already. But
there's a subtle problem lurking in here that gets worse as `head_size` grows, which we'll now
surface deliberately.

### The scaling problem

`key`, `query`, and `value` here are freshly initialized, so `k` and `q` each have roughly unit
variance per entry (this is standard for `nn.Linear` initialization). But `wei = q @
k.transpose(-2,-1)` sums `head_size` individual products to compute each entry — and summing
more independent, unit-variance terms increases variance. Let's measure this directly.


In [27]:
k_demo = torch.randn(B, T, head_size)
q_demo = torch.randn(B, T, head_size)
wei_demo = q_demo @ k_demo.transpose(-2, -1)   # NOTE: no scaling applied yet

print("var(k):  ", k_demo.var().item())
print("var(q):  ", q_demo.var().item())
print("var(wei):", wei_demo.var().item())


var(k):   1.044861912727356
var(q):   1.0700464248657227
var(wei): 17.46897315979004


`k` and `q` start at roughly variance 1, but `wei` comes out with variance close to
`head_size` (16 here) — exactly the "sum of `head_size` unit-variance terms" effect predicted
above. Why does that matter? Because `softmax` is *very* sensitive to the scale of its inputs:


In [28]:
example_logits = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])

print("modest-scale logits  -> softmax:", torch.softmax(example_logits, dim=-1))
print("8x larger logits     -> softmax:", torch.softmax(example_logits * 8, dim=-1))


modest-scale logits  -> softmax: tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
8x larger logits     -> softmax: tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


Multiplying the same logits by 8 makes the softmax output much "peakier" — almost all of the
probability mass collapses onto the single largest entry, converging toward a one-hot vector.
Translated back to attention: if `wei`'s variance is allowed to blow up with `head_size`, the
softmax in our `Head` will saturate, every token will end up attending almost exclusively to
just one other position regardless of what the data actually supports, and gradients through
that near-one-hot softmax become tiny (vanishing) — which makes the model much harder to train,
especially early on. We want `wei` to stay diffuse enough that attention can express genuine
graded preferences, and that gradients can flow.

### The fix: scaled attention

The fix is simple: divide the raw dot products by $\sqrt{\text{head\_size}}$ before the
softmax. Since $\text{Var}(aX) = a^2 \text{Var}(X)$, scaling by $\frac{1}{\sqrt{d}}$ exactly
cancels out the "sum of `d` unit-variance terms" effect, bringing `wei`'s variance back down to
roughly 1 — regardless of how large `head_size` is. This is exactly the "Scaled" in "Scaled
Dot-Product Attention" from the original Transformer paper.


In [29]:
wei_scaled = (q_demo @ k_demo.transpose(-2, -1)) * head_size**-0.5   # <- the fix: * 1/sqrt(d)

print("var(wei), unscaled:", wei_demo.var().item())
print("var(wei), scaled:  ", wei_scaled.var().item())


var(wei), unscaled: 17.46897315979004
var(wei), scaled:   1.0918108224868774


Variance is back down near 1, as intended. Let's apply this fix to our actual `Head`
computation from above and recompute `out`.


In [30]:
# Re-run the real Head computation from earlier, this time with scaling included.
wei = q @ k.transpose(-2, -1) * head_size**-0.5    # <- the only change from before
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

out = wei @ v
out.shape


torch.Size([4, 8, 16])

Same shape as before — scaling doesn't change *what* the operation computes structurally,
only how sharply peaked the resulting attention distribution tends to be. Every attention head
we build from here on (including inside the full GPT) includes this `* head_size**-0.5` term.

### Summary notes on attention

- Attention is a **communication mechanism**: think of it as nodes in a directed graph, each
  looking at the other nodes that point to it and aggregating their information via a weighted
  sum, where the weights are computed from the data itself (not fixed in advance).
- Attention has **no inherent notion of position or space** — it operates over an unordered
  *set* of vectors. (We'll add positional information back in explicitly, later, via
  positional embeddings — without it, attention alone cannot tell "the first token" from "the
  last token.")
- Examples across the **batch dimension are entirely independent** and never communicate with
  each other — only positions within the same sequence attend to each other.
- The triangular mask (`tril`) is what makes this a **decoder** (causal, autoregressive) block,
  appropriate for language modeling, where a position must never see the future. Delete that
  one masking line and you get an **encoder** block, where every position can see every other
  position — used in settings like bidirectional text encoders (e.g. BERT) and NOT used for
  open-ended generation.
- **Self-attention** means the queries, keys, *and* values are all produced from the same
  source `x`. In **cross-attention**, queries still come from `x`, but the keys and values come
  from a *different* source — for example, in a translation model: a French-to-English
  translator's decoder, at every position, asks queries of its own (partially generated)
  English output, but the keys and values it attends to come from the *encoded French
  sentence*:

  ```
  <--------- ENCODE ------------------><--------------- DECODE ----------------->
  les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>
  ```

  The encoder reads the French sentence (no masking — it can see the whole sentence at once)
  and produces key/value vectors summarizing it; the decoder generates the English translation
  one token at a time (causally masked, like our `Head`), using self-attention over what it's
  generated so far *and* cross-attention into the encoded French keys/values to decide what to
  translate next. This notebook only builds the decoder-only, self-attention-only case (which
  is exactly what GPT-style models use), but the same `Head` machinery generalizes directly to
  cross-attention by sourcing K and V from a different tensor than Q.


## Part 8 — Deep Dive: LayerNorm

### Why normalize at all?

Deep networks are easier to train when the values flowing through each layer stay in a
well-behaved numerical range (roughly zero mean, roughly unit variance) — without this,
activations tend to drift toward very large or very small values as they pass through many
layers, which makes gradients unstable and training slow or unstable. Normalization layers
explicitly re-center and re-scale activations at chosen points in the network to keep them
well-behaved, regardless of what came before.

### BatchNorm vs. LayerNorm

**BatchNorm** (the earlier, very widely used normalization layer) normalizes each *feature*
across the *batch*: for feature (channel) $c$, it computes the mean and variance of that one
feature across all examples currently in the batch, and uses those to normalize. This works
well for convolutional nets, but has an awkward dependency on batch size and composition (its
statistics are coupled across examples within a batch, and it needs special handling — running
averages — at evaluation time when you might use a different or smaller batch).

**LayerNorm** (Ba, Kiros & Hinton, 2016) instead normalizes each *example* across its own
*features*: for one token's embedding vector, compute the mean and variance across that one
vector's own entries, and normalize using only those numbers. Every example is normalized
completely independently of every other example in the batch.

$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} \qquad y_i = \gamma \cdot \hat{x}_i +
\beta$$

where $\mu, \sigma^2$ are the mean and variance computed across the feature dimension of a
single example, and $\gamma,\beta$ are learned per-feature scale and shift parameters (so the
network can undo the normalization if that turns out to be useful for a particular feature).

This independence-across-examples is exactly why LayerNorm is the standard choice for
Transformers and sequence models generally: sequence lengths vary, batches are formed somewhat
arbitrarily, and at generation time you may process a single sequence at a time (batch size
1) — none of which causes any trouble for LayerNorm, since it never needs statistics computed
*across* examples in the first place.


In [31]:
class LayerNorm1d:  # a from-scratch implementation, to make the mechanics fully explicit
                     # (used to be called BatchNorm1d in earlier course material — note how
                     # little code actually changes between the two; only WHICH dimension
                     # we reduce over)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
      self.eps = eps
      self.gamma = torch.ones(dim)    # learned per-feature scale, initialized to 1 (no-op)
      self.beta = torch.zeros(dim)    # learned per-feature shift, initialized to 0 (no-op)

  def __call__(self, x):
      # calculate the forward pass
      # dim=1 is the FEATURE dimension here (x has shape (batch, features)) — this is the
      # key difference from BatchNorm, which would reduce over dim=0 (the batch dimension)
      # instead.
      xmean = x.mean(1, keepdim=True)              # mean across features, per example
      xvar = x.var(1, keepdim=True)                 # variance across features, per example
      xhat = (x - xmean) / torch.sqrt(xvar + self.eps)   # normalize to zero mean, unit variance
      self.out = self.gamma * xhat + self.beta            # learned rescale/shift
      return self.out

  def parameters(self):
      return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100)   # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape


torch.Size([32, 100])

In [32]:
# mean and std of ONE FEATURE, computed across all 32 batch examples.
# For LayerNorm this is NOT guaranteed to be (0, 1) -- LayerNorm makes no promise about
# statistics computed across the batch dimension, only within each individual example.
x[:,0].mean(), x[:,0].std()


(tensor(0.1469), tensor(0.8803))

In [33]:
# mean and std of a SINGLE EXAMPLE, computed across its own 100 features.
# This IS guaranteed to be very close to (0, 1) -- that's exactly the normalization
# LayerNorm performs, and it holds independently for every single row.
x[0,:].mean(), x[0,:].std()


(tensor(-9.5367e-09), tensor(1.0000))

Contrast these two cells directly: per-feature-across-batch statistics are *not* standardized
(first cell), but per-example-across-features statistics *are* standardized (second cell) —
the mirror image of what you'd see from BatchNorm. This is the entire conceptual difference
between the two normalization schemes; everything else about the formula is identical.

In production code we won't reimplement this by hand — PyTorch's built-in `nn.LayerNorm(dim)`
does exactly this (and is what we'll use in the Transformer block shortly). It's worth having
built it once from scratch so "LayerNorm" stops being an opaque library call.


## Part 9 — Model Hyperparameters

We now have all the individual pieces we need (attention, scaling, LayerNorm) to build a real
multi-layer Transformer. Before doing so, let's fix the **architecture hyperparameters** we'll
use throughout the rest of the notebook, so that every component we build from here on
(multi-head attention, the feed-forward network, the Transformer block, positional embeddings)
can be demonstrated against consistent, shared values — rather than each one inventing its own
example shapes.

| Hyperparameter | Value | Meaning |
|---|---|---|
| `block_size` | 32 | context window: how many past characters the model conditions on |
| `n_embd` | 64 | size of each token's embedding vector (the "C" dimension throughout) |
| `n_head` | 4 | number of attention heads per Transformer block |
| `n_layer` | 4 | number of stacked Transformer blocks |
| `dropout` | 0.0 | dropout probability (regularization; 0 disables it — our dataset/model are small enough that overfitting isn't the bottleneck) |

Note `block_size` is larger now (32 vs. the toy value of 8 we used earlier) — with real
attention in place, the model can actually make good use of a longer context, so we give it
one. Also note `head_size = n_embd // n_head = 64 // 4 = 16` — this is exactly the
`head_size=16` we already used in all of Part 7's examples; the numbers were chosen to line up.

Separately, we'll detect whether a GPU is available and move tensors there if so — purely a
performance optimization (everything below runs correctly on CPU too, just slower for the full
training run at the end).


In [34]:
block_size = 32   # what is the maximum context length for predictions?
n_embd = 64       # embedding dimension for every token
n_head = 4         # number of attention heads per block
n_layer = 4        # number of Transformer blocks stacked on top of each other
dropout = 0.0      # dropout probability

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)


device: cpu


## Part 10 — Multi-Head Attention

### Why multiple heads instead of one big head?

A single attention head computes exactly one notion of "relevance" between tokens — one
learned way of deciding what to attend to. But there are many different *kinds* of
relationships a model might want to track simultaneously: which character said the previous
line, which word this pronoun refers to, what part of speech typically follows the current
token, and so on. Rather than ask one head to capture all of that at once, we run several
smaller heads **in parallel**, each with its own independent K/Q/V projections (and therefore
free to specialize in a different pattern), and combine their outputs afterward. This is
directly analogous to using multiple filters in a convolutional layer, each free to detect a
different feature.

Concretely: instead of one head with `head_size = n_embd`, we use `n_head` heads each with
`head_size = n_embd // n_head` — so the *total* amount of computation and parameters stays
about the same as one big head would have used, it's just split into `n_head` independent,
narrower attention computations.

### The production `Head` module

This is the same computation as Part 7's `Head`, now packaged as an `nn.Module` so it can be
composed and reused, with two small but important additions:

- **`self.register_buffer('tril', ...)`**: registers `tril` as part of the module's state, so
  it automatically moves with the module when you call `.to(device)` (e.g. onto a GPU) — but
  marks it as NOT a learnable parameter (it's a fixed mask, never updated by the optimizer).
  Plain attributes (as in Part 7's toy version) wouldn't move to the GPU automatically and
  could silently end up on the wrong device.
- **`self.dropout`**: randomly zeroes some attention weights during training, a regularization
  technique that discourages the model from relying too heavily on any single connection. With
  `dropout=0.0` (our setting) this is a no-op, but the architecture includes it because larger
  models / datasets typically need it.


In [35]:
class Head(nn.Module):
    ''' one head of self-attention '''

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # a fixed causal mask, registered as a buffer (not a learnable parameter) so it
        # travels with the module across devices automatically
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)     # (B,T,C) -> really (B,T,head_size); C here means head_size
        q = self.query(x)   # (B,T,C)
        # compute attention scores ("affinities") -- scaled dot-product, as derived in Part 7
        wei = q @ k.transpose(-2,-1) * C**-0.5   # (B, T, C) @ (B, C, T) -> (B, T, T)
        # causal mask: only attend to this position and earlier ones.
        # self.tril[:T, :T] handles sequences shorter than block_size (e.g. during generation,
        # before the context has grown to the full block_size).
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out


In [36]:
# Smoke test: does a single Head run correctly on dummy data of the right shape, and does it
# produce the shape we expect? We use a 2-example "batch" of 10-token sequences, each token
# represented by an n_embd-dimensional vector -- exactly the shape that will flow out of the
# token+position embedding layers later.
head_size = n_embd // n_head
dummy_x = torch.randn(2, 10, n_embd)
test_head = Head(head_size)
print(test_head(dummy_x).shape)   # expect (2, 10, head_size) = (2, 10, 16)


torch.Size([2, 10, 16])


### Combining heads: `MultiHeadAttention`

Run `n_head` independent `Head`s on the same input, concatenate their outputs back together
along the feature dimension (recovering the original `n_embd` width, since `n_head *
head_size = n_embd`), and pass the result through one more learned linear layer (`proj`). That
final projection matters: without it, the heads' outputs would just sit side-by-side with no
way to mix information *between* heads — `proj` lets the model learn to combine the different
heads' findings into a single unified representation.


In [37]:
class MultiHeadAttention(nn.Module):
    ''' multiple heads of self-attention in parallel '''

    def __init__(self, num_heads, head_size):
        super().__init__()
        # nn.ModuleList (not a plain Python list!) so PyTorch correctly tracks every Head's
        # parameters as part of this module (needed for .parameters(), .to(device), etc.)
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)   # mixes information across heads after concat
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # run every head on the same input, then concatenate their outputs along the last
        # (feature) dimension: num_heads tensors of shape (B,T,head_size) -> (B,T,n_embd)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


In [38]:
# Smoke test: n_head heads of head_size each should recombine into n_embd output features.
test_mha = MultiHeadAttention(n_head, head_size)
print(test_mha(dummy_x).shape)   # expect (2, 10, n_embd) = (2, 10, 64)


torch.Size([2, 10, 64])


## Part 11 — The Feed-Forward Network

Attention is a **communication** step: tokens gather information from other tokens. But
attention itself is entirely linear in the values it aggregates (a weighted average is just a
linear combination) — it has no non-linearity, and it never processes a single token's
representation independently of the others. The feed-forward network is the complementary
**computation** step: a small per-token MLP, applied identically and independently at every
position, that lets the model do nonlinear processing on what attention just gathered.

A useful mental model for one Transformer block: *"communicate, then think."* Attention lets
tokens exchange information; the feed-forward network lets each token, now holding richer
information after attention, individually \"think\" about what it just received.

The standard design (introduced in the original Transformer paper, and used essentially
unchanged in GPT-2/3): a linear layer that **expands** the embedding dimension by 4x, a ReLU
nonlinearity, then a linear layer that **projects back down** to the original embedding
dimension. The 4x expansion is simply an empirically-settled convention — giving the
nonlinearity a larger intermediate space to compute in — not a value derived from theory.


In [39]:
class FeedFoward(nn.Module):
    ''' a simple linear layer followed by a non-linearity '''

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),   # expand: n_embd -> 4*n_embd
            nn.ReLU(),                        # nonlinearity
            nn.Linear(4 * n_embd, n_embd),    # project back down: 4*n_embd -> n_embd
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


In [40]:
# Smoke test: shape should be unchanged end-to-end (n_embd in, n_embd out), since this module
# slots into the residual stream alongside attention and must match its shape exactly.
test_ffwd = FeedFoward(n_embd)
print(test_ffwd(dummy_x).shape)   # expect (2, 10, n_embd) = (2, 10, 64)


torch.Size([2, 10, 64])


## Part 12 — Deep Dive: Residual Connections and the Transformer Block

### Why deep networks need residual connections

We're about to stack `n_layer=4` Transformer blocks on top of each other (real GPT models
stack dozens). Naively, deeper networks should be strictly more capable than shallower ones —
but in practice, simply stacking more layers tends to make optimization *harder*, not just
slower: gradients computed via the chain rule through many layers can shrink (vanish) or grow
(explode) as they're multiplied through each layer's Jacobian, and very deep plain
feed-forward stacks were empirically observed to train *worse* than shallower ones, well into
the era of GPUs and good initialization schemes.

**Residual ("skip") connections**, popularized by He et al.'s ResNet (2015), are a
remarkably simple fix: instead of a layer computing $x_{\text{new}} = f(x)$, compute
$x_{\text{new}} = x + f(x)$ — the layer's output is *added* to its input, rather than
replacing it. This creates what's often called a "residual stream" or "gradient highway": even
if $f$'s gradient is tiny or poorly scaled, gradients can still flow backward through the `+`
unimpeded (the gradient of a sum w.r.t. either input is just 1), giving every layer a direct
path to receive a useful learning signal regardless of how deep the stack is.

Concretely: every block in our Transformer **adds** its attention output and its feed-forward
output onto $x$, rather than overwriting it:

```python
x = x + self.sa(...)      # add attention's contribution to the residual stream
x = x + self.ffwd(...)    # add the feed-forward network's contribution
```

### Where does LayerNorm go? Pre-norm vs. post-norm

The original Transformer paper applied LayerNorm *after* the residual addition ("post-norm").
Later work (and what GPT-2 onward uses) applies it *before* each sub-layer instead
("pre-norm"):

$$x = x + \text{Attention}(\text{LayerNorm}(x))$$
$$x = x + \text{FeedForward}(\text{LayerNorm}(x))$$

Pre-norm keeps the residual stream itself free of any normalization, which empirically makes
very deep Transformers noticeably easier to optimize — this is the convention we use here, and
the one used by GPT-2/3 and most modern decoder-only language models.

### Putting it together: the Transformer `Block`

One `Block` = one round of (normalize → attend → add) followed by one round of (normalize →
feed-forward → add). Stack several of these, and you have the computational core of a GPT.


In [41]:
class Block(nn.Module):
    ''' Transformer block: communication followed by computation '''

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)   # the "communication" step
        self.ffwd = FeedFoward(n_embd)                      # the "computation" step
        self.ln1 = nn.LayerNorm(n_embd)   # pre-norm before attention
        self.ln2 = nn.LayerNorm(n_embd)   # pre-norm before the feed-forward network

    def forward(self, x):
        x = x + self.sa(self.ln1(x))      # residual connection around attention
        x = x + self.ffwd(self.ln2(x))    # residual connection around the feed-forward net
        return x


In [42]:
# Smoke test: stack a couple of Blocks (just like nn.Sequential will do for real, shortly)
# and confirm the shape is preserved exactly -- this is required for residual connections to
# even be well-defined (x and f(x) must have the same shape to be added together).
test_blocks = nn.Sequential(Block(n_embd, n_head), Block(n_embd, n_head))
print(test_blocks(dummy_x).shape)   # expect (2, 10, n_embd) = (2, 10, 64), unchanged


torch.Size([2, 10, 64])


## Part 13 — Positional Embeddings

We already noted this back in Part 7's summary, but it's worth confronting directly: attention,
as we've built it, has **no inherent notion of position**. The same `Head` would produce the
exact same per-position outputs if we permuted which token sat where in the sequence (modulo
the causal mask itself) — attention operates over a *set* of vectors and aggregates them by
content-based similarity, not by where they sit. But word order obviously matters
("dog bites man" ≠ "man bites dog") — so we need to inject positional information explicitly,
rather than hope the model infers it from nothing.

### Learned positional embeddings

The approach used here (and in GPT-2/GPT-3): maintain a *second* embedding table, of shape
`(block_size, n_embd)`, where row `t` is a learned vector representing "this token is at
position `t`." Add that position vector directly into the token's own embedding before
anything else happens. (The original Transformer paper instead used a fixed sinusoidal
function of position, requiring no learned parameters at all — both approaches see wide use;
learned embeddings are simpler to implement and are what GPT-style decoder-only models
standardly use.)


In [43]:
# A learned table mapping "position index" -> a learned n_embd-dimensional vector.
# Note this table has block_size rows, NOT vocab_size rows -- it's indexed by WHERE a token
# is in the sequence (0, 1, 2, ..., block_size-1), not by WHICH token it is.
position_embedding_table = nn.Embedding(block_size, n_embd)

T_demo = 10
# torch.arange(T_demo) = [0, 1, 2, ..., 9] -- the position indices for a 10-token sequence
pos_emb_demo = position_embedding_table(torch.arange(T_demo))
print(pos_emb_demo.shape)   # (T, n_embd) = (10, 64) -- ONE vector per position, no batch dim


torch.Size([10, 64])


In [44]:
# token embeddings have shape (B, T, n_embd); position embeddings have shape (T, n_embd).
# Adding them relies on broadcasting: PyTorch automatically repeats the (T, n_embd) position
# tensor across the batch dimension, so every sequence in the batch gets the same positional
# signal added in, while each token's own content embedding stays unique to that token.
tok_emb_demo = torch.randn(2, T_demo, n_embd)   # pretend token embeddings, batch of 2
combined = tok_emb_demo + pos_emb_demo
print(combined.shape)   # (2, 10, 64) -- shape unchanged, position info now mixed in


torch.Size([2, 10, 64])


That's the entire mechanism: from this point on, every downstream layer (attention, the
feed-forward network) receives vectors that already encode *both* "what token is this" and
"where in the sequence is it," summed together into one vector per position. The model is
free to learn to extract either signal — or both — as needed.


## Part 14 — Assembling and Training the Full GPT

Every piece now exists: tokenization, batching, scaled self-attention, multi-head attention,
the feed-forward network, LayerNorm, residual connections wrapping both into a `Block`, and
positional embeddings. This section just **wires them together** into one model and trains it
— nothing here is conceptually new.

### Training hyperparameters

A second group of hyperparameters, distinct from the architecture hyperparameters in Part 9 —
these control the *training process* itself, not the model's structure.


In [45]:
batch_size = 16        # how many independent sequences will we process in parallel?
max_iters = 5000        # total number of training steps
eval_interval = 500      # how often (in steps) to report train/val loss
learning_rate = 1e-3      # AdamW's step size (see Part 5)
eval_iters = 200          # how many batches to average over when estimating loss (next cell)


### Measuring progress reliably: `estimate_loss`

`loss.item()` from a single training batch is noisy — it's computed from just `batch_size`
randomly sampled sequences, so it jumps around step to step even as the model genuinely
improves. `estimate_loss` reports a much more reliable number by averaging the loss over
`eval_iters` separate batches, for *both* the train and validation splits, so we can directly
watch for overfitting (train loss dropping while val loss stalls or rises).

Two details worth understanding, not just copying:

- **`@torch.no_grad()`**: this decorator disables PyTorch's gradient tracking for the whole
  function. We're not calling `.backward()` here, so building up the computation graph needed
  for backprop would be pure wasted memory and compute — `no_grad()` turns that off.
- **`model.eval()` / `model.train()`**: some layers (notably `nn.Dropout`, and `nn.BatchNorm`
  if it were used) behave *differently* depending on whether the model is in training or
  evaluation mode — dropout randomly zeroes activations during training, but should be fully
  "on" (no zeroing) when evaluating, so evaluation results don't depend on dropout's
  randomness. We set `dropout=0.0` for this run, making this particular detail a no-op here —
  but `.eval()`/`.train()` is essential boilerplate any time a model contains dropout or
  batchnorm, so we include the correct pattern even though it doesn't change behavior at our
  current settings.


In [46]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()   # switch dropout/batchnorm (if any) into evaluation behavior
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()   # switch back to training behavior before returning
    return out


### The full model

A few things to notice, each one a direct extension of the original `BigramLanguageModel`
from Part 4:

- **Two embedding tables now, summed together**, exactly as built in Part 13: token identity
  (`token_embedding_table`) plus position (`position_embedding_table`).
- **`self.blocks`**: `n_layer` stacked `Block`s (Part 12), wrapped in `nn.Sequential` so they
  run one after another automatically.
- **`self.ln_f`**: one final LayerNorm after all the blocks, before the output projection —
  standard practice, cleaning up the residual stream's scale before the final linear layer
  reads from it.
- **`self.lm_head`**: a final linear layer projecting from `n_embd` back up to `vocab_size`,
  producing the next-character logits — this is the part of the model that plays the same role
  the single embedding table played in the bigram model, except now it reads from a richly
  contextualized representation instead of a single raw token.
- **`generate` now crops context to the last `block_size` tokens** (`idx_cond = idx[:,
  -block_size:]`) before each forward pass. The bigram model never needed this, because it only
  ever looked at the single most recent token regardless of how long the generated sequence
  got. Now that the model actually uses up to `block_size` tokens of context — and critically,
  the position embedding table only has `block_size` rows defined — we must ensure we never
  feed in more than `block_size` tokens at once, or the position embedding lookup would fail
  for positions beyond what it was ever trained on.

**On the class name:** Karpathy's original notebook keeps this class named
`BigramLanguageModel`, even at this point, as a deliberate joke / teaching device — every line
inside it has by now been completely rewritten, and there's nothing "bigram" about it anymore
(it can condition on up to `block_size` tokens of real context). We rename it here to
`GPTLanguageModel`, since by this point that's a far more accurate name for what it does.


In [47]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)             # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)  # project back up to vocabulary size

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx)                              # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb     # (B,T,C) -- content + position, summed (Part 13)
        x = self.blocks(x)        # (B,T,C) -- n_layer rounds of attend-then-compute (Part 12)
        x = self.ln_f(x)          # (B,T,C) -- final normalization
        logits = self.lm_head(x)  # (B,T,vocab_size) -- next-character logits at every position

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)   # identical to the bigram model's loss

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens -- required now, see explanation above
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [48]:
torch.manual_seed(1337)

model = GPTLanguageModel()
m = model.to(device)   # move all parameters to GPU if available, no-op otherwise

# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


0.209729 M parameters


For reference, that's roughly 0.2M parameters — tiny by modern standards (GPT-2 small is
124M; GPT-3 is 175B), but every architectural idea here is the same one those larger models
use, just at much greater depth and width.


In [49]:
# create a PyTorch optimizer -- AdamW, exactly as derived and used in Part 5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


### Training

Same four-step loop from Part 5 (forward, zero_grad, backward, step), just with `estimate_loss`
called periodically so we can watch both train and validation loss evolve. Expect this cell to
take a few minutes on CPU, faster on GPU.


In [51]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


step 0: train loss 1.8862, val loss 1.9719
step 500: train loss 1.8386, val loss 1.9523
step 1000: train loss 1.7826, val loss 1.9239
step 1500: train loss 1.7620, val loss 1.9050
step 2000: train loss 1.7134, val loss 1.8607


KeyboardInterrupt: 

Compare these loss values to where the bigram model plateaued in Part 5 (loss in the low-2s
at best, with no path to improvement). Train and validation loss should both be considerably
lower here, and should track each other reasonably closely (if validation loss were
*increasing* while training loss kept dropping, that would be the signature of overfitting —
not something we expect to see at this model/dataset scale).


In [ ]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))



Second a loves one angrula, and spity.

COMINIUS:
Aoy may lost.
Grouth, her! I gongen maper. Hell in yet night? Beary, why lafes.
Prevolen him a meturrew as lagdering unge
But when hown woods betselveny ferst in his liked assing hath inder;
Who Khalls dearty both amptagant,
This hold kneed when?
I have you dreayst tarm not, as or slow,
And it bleing such the fides; that while him of this airt.

ROMEO:
Who live help nay Enate. Dayparineor: that made:
And so whath thy 'air.
I mespare last sity bll


This won't read as fluent, meaningful English — at 0.2M parameters and a few thousand training
steps, that would be far too much to expect — but compare it to the bigram model's output from
Part 5: you should see noticeably more English-like structure: real short words, plausible
letter sequences within words, character names followed by colons, line breaks in roughly the
right places. That's the model actually using its attention-derived context, not just
per-character statistics.


## Part 15 — Wrap-Up and Where to Go Next

### The full architecture, recapped

```
tokens (B,T)
   │
   ├─ token_embedding_table[idx]        -> (B,T,n_embd)   "what is this token"
   ├─ position_embedding_table[arange]  -> (T,n_embd)     "where is this token"   (Part 13)
   └─ sum -> x                          -> (B,T,n_embd)
                │
   ┌────────────▼────────────┐  x n_layer  (Part 12)
   │ x = x + MHA(LN(x))       │  <- self-attention, causally masked, scaled    (Parts 7,10)
   │ x = x + FFwd(LN(x))      │  <- per-token nonlinear computation             (Part 11)
   └────────────┬────────────┘
                │
        final LayerNorm (ln_f)
                │
        lm_head: Linear(n_embd -> vocab_size)
                │
            logits (B,T,vocab_size)
```

Every box in that diagram is something we built and individually tested earlier in this
notebook before it was ever assembled into the full model — that incremental-build-then-
-assemble structure is the main thing this version of the notebook changes relative to the
original.

### What we deliberately left out

This is a genuine, working, from-scratch GPT — but it is intentionally minimal. Things present
in production-grade LLMs that are *not* here, each a reasonable next thing to study:

- **Subword tokenization** (BPE / SentencePiece) instead of character-level — see Part 2.
- **Much greater scale**: more layers, wider embeddings, far more training data and compute
  (these are the central subject of "scaling laws" research — empirical relationships between
  model size, data size, compute, and resulting loss).
- **KV-caching** at inference time, so `generate` doesn't recompute attention over the entire
  context from scratch at every new token.
- **Modern architectural variants**: RoPE / ALiBi positional schemes instead of learned
  absolute position embeddings, RMSNorm instead of LayerNorm, gated MLP variants (SwiGLU),
  grouped-query / multi-query attention for inference efficiency, and others — all incremental
  refinements on the same architecture built here, none of which change the core ideas in this
  notebook.
- **Post-training**: everything in this notebook is unsupervised next-token prediction
  ("pretraining"). The instruction-following, chat-style behavior of assistants like ChatGPT
  comes from additional stages (supervised fine-tuning, RLHF, etc.) applied *on top of* a
  pretrained base model essentially like this one, just vastly larger.

### Where to go next

- Andrej Karpathy's [nanoGPT](https://github.com/karpathy/nanoGPT) repository implements
  exactly this model, scaled up, with a real training script (multi-GPU, checkpointing,
  learning rate scheduling) — a natural next step after this notebook.
- The original papers worth reading directly: ["Attention Is All You
  Need"](https://arxiv.org/abs/1706.03762) (Vaswani et al., 2017 — introduces the Transformer),
  the [GPT-2 paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
  (Radford et al., 2019 — the decoder-only, GPT-style architecture used here), and ["Decoupled
  Weight Decay Regularization"](https://arxiv.org/abs/1711.05101) (Loshchilov & Hutter, 2017 —
  AdamW, from Part 5).
- For a mechanistic, "what is actually happening inside these attention heads and MLPs"
  perspective rather than a "how do I build one" perspective, the field of **mechanistic
  interpretability** picks up exactly where this notebook leaves off — reverse-engineering
  what circuits like the ones we just built actually learn to compute once trained.
